Imports 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble         import RandomForestRegressor
from sklearn.model_selection  import train_test_split, cross_val_score
from sklearn.preprocessing    import LabelEncoder
from sklearn.metrics          import mean_absolute_error, mean_squared_error, r2_score
import joblib

Chargement

In [ ]:
df = pd.read_csv("house_rent_clean.csv")
print(f" Dataset chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

Préparation des Features

In [ ]:
# Colonnes utilisées
CATEGORICAL_COLS = ['City', 'Furnishing Status', 'Area Type', 'Tenant Preferred']
NUMERICAL_COLS   = ['BHK', 'Size', 'Bathroom', 'floor_number', 'total_floors']
TARGET           = 'Rent'

print("Features numériques  :", NUMERICAL_COLS)
print("Features catégorielles:", CATEGORICAL_COLS)
print("Cible                :", TARGET)

In [ ]:
# Encodage Label des variables catégorielles
df_enc  = df.copy()
encoders = {}

for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))
    encoders[col] = le
    print(f"  {col:25s} → {len(le.classes_)} classes : {list(le.classes_)}")

print("\n Encodage terminé")

In [ ]:
# Matrice de features et vecteur cible
feature_cols = NUMERICAL_COLS + CATEGORICAL_COLS

X = df_enc[feature_cols].fillna(-1)
y = df_enc[TARGET]

print(f"Forme de X : {X.shape}")
print(f"Forme de y : {y.shape}")
print(f"\nStatistiques du loyer cible :")
print(y.describe().round(0).to_string())

Split Train / Test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train : {X_train.shape[0]:,} exemples  ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test  : {X_test.shape[0]:,} exemples   ({X_test.shape[0]/len(X)*100:.0f}%)")

# Vérification de la distribution des loyers dans les deux sets
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y_train, bins=40, color='#4F8EF7', edgecolor='white', alpha=0.8)
axes[0].set_title(f'Loyers — Train ({len(y_train):,} lignes)')
axes[0].set_xlabel('Loyer (₹/mois)')

axes[1].hist(y_test, bins=40, color='#50C878', edgecolor='white', alpha=0.8)
axes[1].set_title(f'Loyers — Test ({len(y_test):,} lignes)')
axes[1].set_xlabel('Loyer (₹/mois)')

plt.tight_layout()
plt.show()

Entraînement du Modèle 

In [ ]:
# Paramètres du RandomForest
params = {
    'n_estimators'    : 200,
    'max_depth'       : 15,
    'min_samples_split': 5,
    'min_samples_leaf' : 2,
    'random_state'    : 42,
    'n_jobs'          : -1,
}

print("Paramètres du modèle :")
for k, v in params.items():
    print(f"  {k:25s} = {v}")

In [1]:
%%time
# Entraînement
model = RandomForestRegressor(**params)
model.fit(X_train, y_train)

print("✅ Modèle entraîné !")
print(f"   Nombre d'arbres : {model.n_estimators}")
print(f"   Profondeur max  : {model.max_depth}")

NameError: name 'RandomForestRegressor' is not defined

Évaluation du Modèle

In [2]:
# Prédictions sur le set de test
y_pred = model.predict(X_test)

# Métriques
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test.values - y_pred) / (y_test.values + 1e-5))) * 100

print("╔══════════════════════════════════════════╗")
print("║         MÉTRIQUES DU MODÈLE              ║")
print("╠══════════════════════════════════════════╣")
print(f"║  MAE  (Erreur absolue moyenne) : ₹{mae:>8,.0f} ║")
print(f"║  RMSE (Erreur quadratique)     : ₹{rmse:>8,.0f} ║")
print(f"║  R²   (Coeff. détermination)   : {r2:>10.4f} ║")
print(f"║  MAPE (Erreur en %)            : {mape:>9.2f}% ║")
print("╚══════════════════════════════════════════╝")

NameError: name 'model' is not defined

In [3]:
# Validation croisée (5 folds)
print("Validation croisée (5-fold) en cours...")
cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2', n_jobs=-1)

print(f"\nR² par fold : {[f'{s:.4f}' for s in cv_scores]}")
print(f"R² moyen    : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

Validation croisée (5-fold) en cours...


NameError: name 'cross_val_score' is not defined

Importance des Variables

In [4]:
# Importance des features
importances = model.feature_importances_
fi_df = pd.DataFrame({
    'Feature'    : feature_cols,
    'Importance' : importances
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(fi_df)))
bars = ax.barh(fi_df['Feature'], fi_df['Importance'], color=colors, edgecolor='white')

for bar, val in zip(bars, fi_df['Importance']):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

ax.set_title("Importance des variables (RandomForest)", fontsize=13)
ax.set_xlabel("Importance relative")
plt.tight_layout()
plt.show()

print("\nTop 3 features les plus importantes :")
for _, row in fi_df.tail(3).iloc[::-1].iterrows():
    print(f"  {row['Feature']:25s} : {row['Importance']:.4f}")

NameError: name 'model' is not defined

Valeurs Réelles vs Prédites

In [5]:
# Échantillon de 500 points pour la lisibilité
idx    = np.random.RandomState(42).choice(len(y_test), size=min(500, len(y_test)), replace=False)
y_real = y_test.iloc[idx].values
y_pr   = y_pred[idx]
resid  = y_real - y_pr

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Réel vs Prédit ───────────────────────────
max_val = max(y_real.max(), y_pr.max())
axes[0].scatter(y_real, y_pr, alpha=0.4, s=20, color='#4F8EF7')
axes[0].plot([0, max_val], [0, max_val], 'r--', linewidth=1.5, label='Prédiction parfaite')
axes[0].set_title("Valeurs réelles vs prédites")
axes[0].set_xlabel("Loyer réel (₹)")
axes[0].set_ylabel("Loyer prédit (₹)")
axes[0].legend()
fmt = mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}k')
axes[0].xaxis.set_major_formatter(fmt)
axes[0].yaxis.set_major_formatter(fmt)

# ── Résidus ──────────────────────────────────
axes[1].hist(resid, bins=40, color='#F4845F', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_title("Distribution des résidus")
axes[1].set_xlabel("Résidu = Réel − Prédit (₹)")
axes[1].xaxis.set_major_formatter(fmt)

plt.tight_layout()
plt.show()

print(f"Résidu moyen : ₹{resid.mean():,.0f}")
print(f"Résidu std   : ₹{resid.std():,.0f}")

NameError: name 'np' is not defined

Prédiction Interactive

In [6]:
def predict_rent(bhk, size, bathroom, city, furnishing, area_type, tenant,
                 floor_number=1, total_floors=5):
    """
    Prédit le loyer pour un logement donné.

    Exemple :
        loyer = predict_rent(
            bhk=2, size=800, bathroom=2,
            city='Mumbai', furnishing='Semi-Furnished',
            area_type='Super Area', tenant='Family'
        )
    """
    input_dict = {
        'BHK'              : [bhk],
        'Size'             : [size],
        'Bathroom'         : [bathroom],
        'floor_number'     : [floor_number],
        'total_floors'     : [total_floors],
        'City'             : [city],
        'Furnishing Status': [furnishing],
        'Area Type'        : [area_type],
        'Tenant Preferred' : [tenant],
    }
    row = pd.DataFrame(input_dict)

    # Encodage des catégorielles avec les encoders entraînés
    for col in CATEGORICAL_COLS:
        le  = encoders[col]
        val = str(row[col][0])
        val = val if val in le.classes_ else le.classes_[0]
        row[col] = le.transform([val])

    X_input = row[feature_cols].fillna(-1)
    pred    = model.predict(X_input)[0]
    return round(max(0, pred), 2)


print("Fonction predict_rent() prête")

Fonction predict_rent() prête


In [7]:
# ── Exemples de prédictions ───────────────────
exemples = [
    dict(bhk=1, size=350, bathroom=1, city='Chennai',   furnishing='Unfurnished',    area_type='Carpet Area', tenant='Bachelor'),
    dict(bhk=2, size=800, bathroom=2, city='Mumbai',    furnishing='Semi-Furnished', area_type='Super Area',  tenant='Family'),
    dict(bhk=3, size=1500, bathroom=3, city='Bangalore', furnishing='Furnished',     area_type='Built Area',  tenant='Family'),
    dict(bhk=4, size=2500, bathroom=4, city='Delhi',     furnishing='Furnished',     area_type='Super Area',  tenant='Family'),
]

print(f"{'Config':50s} | {'Loyer prédit':>15s}")
print("─" * 68)
for ex in exemples:
    loyer = predict_rent(**ex)
    label = f"{ex['bhk']}BHK {ex['size']}sqft {ex['city']} ({ex['furnishing'][:4]}...)"
    print(f"{label:50s} | ₹{loyer:>12,.0f}")

Config                                             |    Loyer prédit
────────────────────────────────────────────────────────────────────


NameError: name 'pd' is not defined

In [8]:
# ── Prédiction personnalisée — modifiez ces valeurs ! ────────
bhk        = 2
size       = 900          # superficie en sq ft
bathroom   = 2
city       = 'Hyderabad'  # Mumbai, Delhi, Bangalore, Chennai, Kolkata, Hyderabad
furnishing = 'Semi-Furnished'  # Furnished / Semi-Furnished / Unfurnished
area_type  = 'Super Area'      # Super Area / Built Area / Carpet Area
tenant     = 'Family'          # Family / Bachelor / Bachelor/Family
floor_num  = 3
total_flr  = 8

loyer_estime = predict_rent(
    bhk=bhk, size=size, bathroom=bathroom,
    city=city, furnishing=furnishing,
    area_type=area_type, tenant=tenant,
    floor_number=floor_num, total_floors=total_flr
)

# Comparaison avec la médiane de la ville
mediane_ville = df[df['City'] == city]['Rent'].median()

print("╔══════════════════════════════════════════╗")
print("║         ESTIMATION DE LOYER              ║")
print("╠══════════════════════════════════════════╣")
print(f"║  Logement : {bhk}BHK, {size} sq ft, {bathroom} sdb         ║")
print(f"║  Ville    : {city:<33s}║")
print(f"║  Meublé   : {furnishing:<33s}║")
print("╠══════════════════════════════════════════╣")
print(f"║   Loyer estimé  : ₹{loyer_estime:>8,.0f} /mois       ║")
print(f"║   Médiane ville : ₹{mediane_ville:>8,.0f} /mois       ║")
print(f"║   Écart         : ₹{loyer_estime - mediane_ville:>+8,.0f}             ║")
print("╚══════════════════════════════════════════╝")

NameError: name 'pd' is not defined

Sauvegarde du Modèle

In [9]:
# Sauvegarder le modèle et les encoders
joblib.dump(model,    "model.joblib")
joblib.dump(encoders, "encoders.joblib")

# Vérification
import os
print("Fichiers sauvegardés :")
for f in ["model.joblib", "encoders.joblib"]:
    size_kb = os.path.getsize(f) / 1024
    print(f"   {f:<25s} ({size_kb:.1f} KB)")

NameError: name 'joblib' is not defined

In [10]:
# Recharger et vérifier
model_loaded    = joblib.load("model.joblib")
encoders_loaded = joblib.load("encoders.joblib")

# Test rapide
test_pred = model_loaded.predict(X_test[:5])
orig_pred = model.predict(X_test[:5])

print("Vérification du rechargement (5 premières prédictions) :")
for i, (p1, p2) in enumerate(zip(orig_pred, test_pred)):
    match = "✅" if abs(p1 - p2) < 1 else "❌"
    print(f"  [{i}] Original: ₹{p1:,.0f}  |  Rechargé: ₹{p2:,.0f}  {match}")

NameError: name 'joblib' is not defined